# 02 — Data Cleaning

**Objetivo:** Corrigir todos os problemas identificados no profiling e construir o dataframe master que será usado nas análises.

**Transformações aplicadas:**
- Conversão de tipos (datas, valores monetários, horas)
- Correção de nomes de colunas
- Criação de colunas derivadas (dia da semana, mês, flag de entrega com problema)
- Join entre pedidos, clientes e motoristas

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loader import load_orders, load_customers, load_products, load_drivers, load_order_items
from src.preprocessing import clean_orders, clean_products, clean_drivers, build_master

pd.set_option("display.max_columns", None)

## 1. Limpeza individual de cada tabela

In [ ]:
orders   = clean_orders(load_orders())
products = clean_products(load_products())
drivers  = clean_drivers(load_drivers())
customers = load_customers()
order_items = load_order_items()

print("orders:", orders.shape)
print("products:", products.shape)
print("drivers:", drivers.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)

## 2. Verificação dos tipos após limpeza

In [ ]:
print("=== ORDERS ===")
print(orders.dtypes)
print("\nAmostra:")
display(orders[["date", "order_amount", "delivery_hour", "day_of_week", "month", "has_missing"]].head())

In [ ]:
print("=== PRODUCTS ===")
print(products.dtypes)
display(products.head(3))

## 3. Construção do dataframe master

In [ ]:
master = build_master(orders, customers, drivers)

print(f"Master dataframe: {master.shape[0]:,} linhas | {master.shape[1]} colunas")
display(master.head())

## 4. Validação final

In [ ]:
print("Nulos no master:")
nulls = master.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "Nenhum valor nulo.")

print("\nIntervalo de datas:")
print(f"  De: {master['date'].min().date()}")
print(f"  Até: {master['date'].max().date()}")

print("\nReceita total:")
print(f"  R$ {master['order_amount'].sum():,.2f}")

print("\nTaxa geral de itens faltando:")
print(f"  {master['has_missing'].mean()*100:.1f}% dos pedidos")

## 5. Exportar dados limpos

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

master.to_parquet("../data/processed/master.parquet", index=False)
products.to_parquet("../data/processed/products.parquet", index=False)
order_items.to_parquet("../data/processed/order_items.parquet", index=False)

print("Dados exportados para data/processed/")